# import

In [1]:
import json

filename = 'test_data.json'
try:
    file_handle = open(filename, 'r', encoding='UTF8')
    cursor = json.load(file_handle)
except Exception as e:
    print(e)


In [2]:
import pandas as pd

df = pd.DataFrame(cursor)

# HGramm: get_textarea  
HGramm 기반 문장의 표제어 확률과 LLM 기반 연속된 문장 간 의미 유사도로 본문 영역(textarea) 선출합니다.
  
**동작 순서**  
1. HGramm으로 각 문장이 표제어에 가까운지, 본문 내 완결된 문장에 가까운지 판단합니다. (is_head 예측. [0,1] float값)
2. is_head 값을 window 크기 4 mean pooling으로 평탄화한 뒤, mean pooling 값이 크게 낮아지거나 커지는 구간을 본문-비본문 전환 영역으로 정의합니다.
3. 본문-비본문 전환 영역에 해당되는 연속된 4(=mean pooing window size)개의 문장을 추출합니다. 이 문장들을 LLM으로 벡터 임베딩합니다.
4. 연속된 문장들의 코사인 유사도를 구해, 코사인 유사도가 급격히 낮아지는 곳을 본문-비본문 전환점으로 정의합니다.
5. 본문-비본문 전환점을 기준으로 textarea를 추출합니다.

In [3]:
from HGramm import HGramm

hg = HGramm()

c:\Users\psjoy\anaconda3\envs\mecab_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# llm 경로는 local path나 hugging face url을 사용합니다.
# llm_path는 SentenceTransformer(llm_path)로 모델을 불러올 때 사용됩니다.
llm_path = "C:/Users/psjoy/huggingface_models/HyperCLOVAX-SEED-Text-Instruct-0.5B"
hg.set_llm(llm_path=llm_path)

# mecab 사용을 위한 dicpath 경로를 설정합니다
mecab_path = 'C:/mecab/mecab-ko-dic'
hg.set_mecab(mecab_path=mecab_path)

No sentence-transformers model found with name C:/Users/psjoy/huggingface_models/HyperCLOVAX-SEED-Text-Instruct-0.5B. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 218/218 [00:00<00:00, 13631.47it/s]


gpu에 모델 할당 완료!


In [ ]:
from time import time
start = time()

text_col = 'text'
id_col = 'url'
df_textarea = hg.get_textarea(cursor, text_col=text_col, id_col=id_col)

end= time()
print(f'수행 시간: {end-start} sec')
# 문서 100개에서 textarea 추출에 5.4초 소요

수행 시간: 5.450732946395874 sec


In [ ]:
df_textarea

,text,url,is_head,textarea_range,textarea
0,"[LG유플러스, 서울 지하철 9호선 1·2·3단계 LTE-R 구축 완료, 경기·인천...",http://www.cnbnews.com/news/article.html?no=76...,"[0.9538815298700234, 0.9857258919267602, 0.953...","[4, 16]",[서울 강서구 서울시메트로9호선 사옥에서 열린 구축 완료 보고회에서 박성주 서울메트...
1,"[평창군, 치유의 숲 시범운영...이용자 97% 만족, 2025.12.09 17:1...",http://www.cnbnews.com/news/article.html?no=76...,"[0.9683629153506383, 0.9955007753804112, 0.786...","[3, 9]",[(CNB뉴스=신규성 기자) 강원 평창군이 조성한 ‘평창 치유의 숲’이 시범운영 기...
2,"[고성군, 공공 배달앱 ‘먹깨비’로 지역경제 선순환 촉진, 2025.12.09 17...",http://www.cnbnews.com/news/article.html?no=76...,"[0.9800190954792404, 0.9955007753804112, 0.794...","[3, 8]",[(CNB뉴스=신규성 기자) 평화 경제 거점도시 강원 고성군이 지역경제 활성화를 위...
3,"[신라면세점, 박대윤 초청 메이크업 클래스, 2025.12.10 17:08:10, ...",http://www.cnbnews.com/news/article.html?no=76...,"[0.6940779748020067, 0.9955007753804112, 0.005...","[2, 8]",[신라면세점이 메이크업 아티스트 박태윤 실장을 초청해 뷰티 클래스를 진행한다. (사...
4,"[엔씨소프트 리듬게임 ‘러브비트’, 오리지널 음원 발매 기념 이벤트 진행, 2025...",http://www.cnbnews.com/news/article.html?no=76...,"[0.9515932728257773, 0.9955007753804112, 0.003...","[2, 10]",[엔씨소프트는 PC 리듬액션 게임 ‘러브비트’가 오리지널 음원을 발매하고 관련 이벤...
...,...,...,...,...,...
96,"[경복대학교 유아교육학과, 남양주시 상상누리터와 '영어뮤지컬프로그램' 성료, 저출산...",http://www.cnbnews.com/news/article.html?no=76...,"[0.9825452181228299, 0.8774161740159986, 0.995...","[3, 13]",[유아교육학과 전공동아리 ‘아해다솜’이 남양주시 상상누리터 진접장승센터에서 초등학생...
97,"[경복대학교 공연예술학과, 유아 대상 참여형 창작아동극 'Willow-버들이' 순회...",http://www.cnbnews.com/news/article.html?no=76...,"[0.9626670791580787, 0.7900725357779789, 0.995...","[4, 18]",[경복대학교 공연예술학과 전공심화과정 학생들이 제작한 참여형 창작아동극 'Willo...
98,"[인천시, 공촌정수장 소독 방식 개선...소독 냄새 줄이고 안전성 높여, 차아염소산...",http://www.cnbnews.com/news/article.html?no=76...,"[0.9646254396499215, 0.9076727388656821, 0.995...","[3, 12]",[인천광역시 상수도사업본부 공촌정수사업소는 수돗물 소독제로 사용해 온 염소(Cl2)...
99,"[인천시, 환경오염물질 배출사업장 ‘민간전문가 기술지원’으로 환경관리 강화, 대기·...",http://www.cnbnews.com/news/article.html?no=76...,"[0.9782963275959317, 0.9590893207188097, 0.995...","[3, 10]",[인천광역시는 관내 산업단지 내 환경오염물질 배출사업장을 대상으로 민간전문가와 함께...


In [13]:
df_textarea.loc[99,'textarea']

['인천광역시는 관내 산업단지 내 환경오염물질 배출사업장을 대상으로 민간전문가와 함께 무료 환경관리 기술지원을 실시했다고 밝혔다.',
 '인천시는 현재 국가산업단지 3개소, 일반산업단지 11개소, 첨단산업단지 1개소 등 총 1만3,769개 업체가 입주해 있으며 대기·수질 배출업소는 2,158개소에 이른다.',
 '인천시는 올해 배출업소 총 2,158개소 중 대기배출시설 264개소, 수질(폐수) 배출시설 233개소, 대기·수질 통합배출시설 858개소 등 총 1,356개소를 지도·점검했으며 지난달 말 기준 125건의 위반사항을 적발했다.',
 '최근 산업단지 노후화와 인접한 고층 주거지역 증가로 대기 및 폐수 관리시설의 환경오염 우려가 커지고 이에 따라 시민들의 민원도 지속적으로 발생하고 있다.',
 '인천시는 이러한 문제를 해결하기 위해 민간 환경기술 전문가의 기술지원을 통해 영세배출사업장의 운영관리 능력 강화와 현장 애로 해소에 나섰다.',
 '올해 하반기 환경관리 기술지원은 행정처분 이력 또는 민원 발생 업체를 중심으로 대기 5곳, 수질 3곳 등 총 8개 업체를 대상으로 추진되었다.',
 '기술지원은 인천녹색환경지원센터 내 10년 이상 경력을 갖춘 전문가가 직접 사업장을 방문해 대기·폐수시설 운영관리 상태 점검, 환경 관련법 위반 가능성 확인, 민원 발생 요인 및 구조적 문제 진단, 현장의 어려움 청취, 개선 방안 제시 및 사후관리 지원 등 현장 중심의 맞춤형 컨설팅으로 진행되었다.',
 '정승환 시 환경국장은 “인천은 산업단지와 인접한 주거지역이 공존하는 도시로, 오염물질 배출시설의 관리 수준이 시민 건강과 환경에 직접적으로 영향을 미친다”며 “환경오염물질 배출사업장에 대한 지속적인 점검과 전문 기술지원 강화로 보다 깨끗하고 안전한 생활환경을 조성해 가겠다”고 말했다.']